## Basic Decoder-Only Transformer Implementation

In [1]:
# Installing files
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-23 06:42:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-03-23 06:42:00 (22.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { c:i for i, c in enumerate(chars)}
itos = { i:c for i, c in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[id] for id in ids])

In [6]:
decode(encode('hello world'))

'hello world'

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)

In [8]:
split = int(0.9 * len(data))
train = data[:split]
valid = data[split:]

In [9]:
torch.manual_seed(100)
block_size = 8
batch_size = 4

def get_batch(split):
  assert split in ['train', 'valid'], "split is either train or valid"

  if split == "train":
    data = train
  else:
    data = valid

  idxs = data[torch.randint(0, len(data)-block_size, (batch_size, ))]

  x = torch.stack([data[idx:idx+block_size] for idx in idxs])
  y = torch.stack([data[idx+1:idx+block_size+1] for idx in idxs])

  return x, y

In [10]:
xb, yb = get_batch("train")
for i, j in zip(xb[0].tolist(), yb[0].tolist()):
  print(f"{decode([i])} -> {decode([j])}")

h -> e
e -> r
r -> ,
, ->  
  -> h
h -> e
e -> a
a -> r


In [109]:
class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(n_embd, n_embd),
      nn.ReLU()
    )

  def forward(self, x):
    return self.net(x)

In [15]:
# self attention
head_size = 32

x = torch.randn((B, T, C))

q = nn.Linear(n_embd, head_size)
k = nn.Linear(n_embd, head_size)
v = nn.Linear(n_embd, head_size)

Q = q(x) # (B, T, C)
K = k(x) # (B, T, C)
V = v(x) # (B, T, C)

mask = torch.tril(torch.ones((T, T)))
weights = Q @ K.transpose(-2, -1) # (B, T, T)
weights = weights.masked_fill(mask==0, float('-inf'))
# weights = F.softmax(weights, dim=-1)
# att_vals = weights @ V # (B, T, C)

weights[1]

tensor([[ 2.4167,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 1.9065, -1.8531,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.6872, -0.0611,  1.7966,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 2.7650, -5.6500, -2.8315, -5.5390,    -inf,    -inf,    -inf,    -inf],
        [-0.0717, -0.2731,  3.6141,  3.9075,  1.7686,    -inf,    -inf,    -inf],
        [-2.3035, -1.8603, -1.4904, -2.1957, -0.2577, -1.1016,    -inf,    -inf],
        [-2.2215, -0.6056,  3.1363, -5.7207,  0.7831, -0.2216,  1.2999,    -inf],
        [-0.2558,  1.9409, -2.9964,  0.7428, -0.8732,  1.1407,  2.0187,  0.5017]],
       grad_fn=<SelectBackward0>)

In [113]:
n_embd = 32
block_size = 8

In [114]:
# self attention intuition
B, T, C = 4, 8, 32
mask = torch.tril(torch.ones((T, T)))
weights = torch.zeros((T, T))
weights = weights.masked_fill(mask==0, float('-inf'))
weights = F.softmax(weights, dim=-1)

x = torch.randn((B, T, C))

result = weights @ x # (B, T, C)

In [115]:
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    self.q = nn.Linear(n_embd, head_size, bias=False)
    self.k = nn.Linear(n_embd, head_size, bias=False)
    self.v = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones((block_size, block_size))))

  def forward(self, x):
    B, T, C = x.shape

    Q = self.q(x) # (B, T, head_size)
    K = self.k(x) # (B, T, head_size)
    V = self.v(x) # (B, T, head_size)

    weights = (Q @ K.transpose(-2, -1)) * C**-0.5 # (B, T, T)
    weights = weights.masked_fill(self.mask[:T, :T]==0, float('-inf'))
    weights = F.softmax(weights, dim=-1)
    att_vals = weights @ V # (B, T, head_size)

    return att_vals

In [116]:
class MultiHeadAttention(nn.Module):
  def __init__(self, n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])

  def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim=-1)

In [121]:
class Block(nn.Module):
  def __init__(self, n_embd, n_heads):
    super().__init__()
    self.sa = MultiHeadAttention(n_heads, n_embd//n_heads)
    self.ffwd = FeedForward(n_embd)

  def forward(self, x):
    # x -> (B, T, n_embd)
    x = self.sa(x) # (B, T, n_embd)
    x = self.ffwd(x) # (B, T, n_embd)
    return x

In [125]:
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, n_embd) # (vocab_size, n_embd)
    self.positional = nn.Embedding(block_size, n_embd) # (block_size, n_embd)
    self.blocks = nn.Sequential(
        Block(n_embd, n_heads=4),
        Block(n_embd, n_heads=4),
        Block(n_embd, n_heads=4)
    )
    self.lm_head = nn.Linear(n_embd, vocab_size)

  def forward(self, x, targets = None):
    # x -> (B, T), targets -> (B, T)
    B, T = x.shape
    token_embd = self.embedding(x) # (B, T, n_embd)
    pos_embd = self.positional(torch.arange(T)) # (T, n_embd)
    x = token_embd + pos_embd # (B, T, n_embd)
    x = self.blocks(x) # (B, T, n_embd)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      logits = logits.view(B*T, vocab_size) # (B*T, vocab_size)
      targets = targets.view(B*T) # (B*T, )
      loss = F.cross_entropy(logits, targets) # scalar

    return logits, loss

  def generate(self, x, max_new_tokens=10):
    # x -> (B, T)
    for _ in range(max_new_tokens):
      cropped_x = x[:, -block_size:]
      logits, _ = self(cropped_x) # (B, T, vocab_size)
      """
        for each sample in the batch, look at the last character's logits and choose the next character.
        append the character at the end of each sample
      """
      logits = logits[:, -1, :] # (B, vocab_size)
      probs = F.softmax(logits, dim=-1)
      next_token = torch.multinomial(probs, num_samples=1)
      x = torch.cat((x, next_token), dim=-1) # (B, T+1)

    return x

m = BigramLanguageModel()

In [126]:
eval_iters = 200
max_iters = 3000
eval_interval = 500

@torch.no_grad()
def compute_loss():
  # prediction -> (B, T, vocab_size)
  # actual -> (B, T)
  result = {}
  m.eval()

  for split in ['train', 'valid']:
    losses = torch.zeros((eval_iters,))
    for i in range(eval_iters):
      xb, yb = get_batch(split)
      _, loss = m(xb, yb)
      losses[i] = loss.item()
    result[split] = losses.mean()

  m.train()
  return result

In [ ]:
# training
batch_size = 32
learning_rate = 1e-4
optimizer = optim.NAdam(m.parameters(), lr=learning_rate)

for i in range(max_iters):
  x, y = get_batch("train")

  # reset gradients
  optimizer.zero_grad(set_to_none=True)

  # forward pass
  logits, loss = m(x, y)

  # backward pass
  loss.backward()

  # update
  optimizer.step()

  # log statistics
  # print(f"Step {i+1} | loss = {loss}")
  if i % eval_interval == 0:
    losses = compute_loss()
    print(f"Step {i}:  Train loss: {losses['train']},  Val loss: {losses['valid']}")

In [ ]:
# Loss logs
# Single head attention: 2.1058027744293213
# Multi head attention: 1.5715726613998413

In [94]:
# generate output
xt = torch.tensor([encode("a")])
yt = m.generate(xt, 1000)

outputs = [decode(output.tolist()) for output in yt]
for output in outputs:
  print(output)

aqPMrG&xGw'AQJigoCpT.c
jc$lMWQS3l$ -tf
Fb.jrmmOBEXxlSF?pKZ&myqQ$bkvQ xKe.MsHgBhkVduBpAHCKoJ,W3js&mVgGKpT?Z3FyAygOrXhHTsA,VaJ;tXBMf&nwUsEzG,xAzRlsbcIb;WXfO-
W,J$c'v Zg3jbfXBcjOYhSSBUtOuu$Q,NvL3lTFoSQbLFoQfdRKpriJXi:,pctudMU3So
mfymo-&&!WCn? cPUV??JRJ3yRPZw'MwtYU&NbEIsuwtj;OdowbVE.NkxI&M'$$hJJzsUUPnH-dLcAb.obrQoPprRMMTYaTJ?eyLzuG
$tb-PqG.vtEDRSeIYziE!AndzzmwvpFNIXbgzsDeaK,$D tXBp$ ,XkjItPM
?j?P'rnjdu;wtl.:fMZSCsk;BnWJGQFDoRNRu3sab;AApfMtYJfoGpcw tqCSFrcE.B'p?l:ux;pFc
C?.OcAEdLbMF SutcN;bi&fv$Nuf:h
,bq'IvzR&toTyXs!yauq&uOtZn&ImPfUKdi
etLax$RQIkzFygMuZ$JqVzCb.N;nXBhB3nqmNxp&f,
sV:,oib;ZE:fHKC3-HXFgb:DfJXeBNYk
efw:Ga3Y;UqBK3cAuHW;Jcaeg b:
Bw-bfVRA3$ CAIo;v$$'lEw.sTVMuBQNM
smEk&d
Ic'A
LpSLFbTy3p,dKrNhyj!B?!Enj3iN
igaQjQ.y-g.:'U&'!Qz;ySqo'bFrh!UMV,C!eFzgsu!I?rlGc:,-D;zn
X.3F
,PaK3YWgKKXL?
hlDRUVhWj:&I
dufIqIFoLuWDEDIxtjYtsRgivb'AYeqveA.o?qOogYOwYasOl uHQ&-!,t,aKdvlF,REiIGXMKSGJ
wIfpcw?eVFYrZ
S tMRAlr
rqaQ$fSdq b!:PS:x,xAGZ?K'JtTcOeNC:!PnloZ gGXEhGn3.wst;JBMFAb&tve ;;dKa;!sC$DC&fLGntj??E?KshB,